In [2]:
import sys
import os
import numpy as np
import tensorflow as tf
from sklearn.metrics import accuracy_score
import gc

# 1. Add project root to path so we can import 'src'
project_root = os.path.abspath(os.path.join(os.getcwd(), '..'))
if project_root not in sys.path:
    sys.path.append(project_root)

from src.data.loader import GliomaDataLoader
from src.models.cvae import CVAE
from src.utils.config_manager import ConfigManager
from src.notebooks.experiment_utils import setup_experiment

# 2. Setup experiment from config
config_name = 'exp_104'
config, output_dir = setup_experiment(config_name)
actual_n_features = loader.X.shape[1]  # Get from loaded data
# Load paths from config
DATA_PATH = config['paths']['processed_file']
SPLIT_FILE = config['paths']['splits_file']
MODEL_DIR = config['paths']['models_cvae_dir']

print(f" Config loaded: {config_name}")
print(f"Data path: {DATA_PATH}")
print(f"Model weights dir: {MODEL_DIR}")

# Build CVAE config from centralized exp_101.yaml
cvae_cfg = config['models']['cvae']

cvae_config = {
    'input_shape': (actual_n_features, 1),
    'num_classes': 6,
    'latent_dim': cvae_cfg['latent_dim'],
    'loss_weights': {
        'beta': cvae_cfg['loss_weights']['beta'],
        'alpha': cvae_cfg['loss_weights']['alpha']
    },
    'encoder': {
        'filters': cvae_cfg['encoder']['filters'],
        'kernels': cvae_cfg['encoder']['kernels'],
        'strides': cvae_cfg['encoder']['strides'],
        'use_global_pooling': False,
        'use_skip_gap': False
    },
    'decoder': {
        'initial_filters': cvae_cfg['decoder']['initial_filters'],
        'use_dense_projection': True,
        'filters': cvae_cfg['decoder']['filters'],
        'kernels': cvae_cfg['decoder']['kernels'],
        'strides': cvae_cfg['decoder']['strides']
    }
}

print(f"CVAE Config loaded from {config_name}:")
print(f"  - Latent dim: {cvae_cfg['latent_dim']}")
print(f"  - Beta: {cvae_cfg['loss_weights']['beta']}")
print(f"  - Alpha: {cvae_cfg['loss_weights']['alpha']}")

# 3. Initialize Data Loader
print("\nInitializing Loader...")
loader = GliomaDataLoader.from_config(
    h5_data_path=DATA_PATH,
    config=config,
    memory_mapped=False
)

print(f" Loader initialized. Total spectra: {loader.X.shape[0]}")

# 4. Setup Results Dictionary
cbr_results = {
    'Classification_by_Recon': []
}

print("\n Starting Generative Inference: Classification by Reconstruction...")

# 5. Main Inference Loop
for fold_data in loader:
    fold_name = fold_data['fold_name']
    print(f"\n{'='*60}")
    print(f" Evaluating {fold_name}")
    print(f"{'='*60}")
    
    # Unpack Real Validation Data
    X_val = fold_data['val'][0].astype(np.float32)
    y_val = fold_data['val'][1]
    
    # Fix 1-indexed to 0-indexed if necessary
    if np.min(y_val) == 1:
        y_val = y_val - 1
        
    X_val_exp = np.expand_dims(X_val, axis=-1) # Shape: (N, actual_n_features, 1)
    n_samples = X_val.shape[0]
    
    # --- MODEL INITIALIZATION ---
    tf.keras.backend.clear_session()
    cvae = CVAE(cvae_config)
    cvae.compile(optimizer='adam')
    
    # Initialize all tensor shapes with a dummy batch
    dummy_x = tf.zeros((2, actual_n_features, 1))
    dummy_y = tf.convert_to_tensor(tf.keras.utils.to_categorical([0, 1], num_classes=6), dtype=tf.float32)
    cvae.train_step((dummy_x, dummy_y))
    
    # Load the specific weights for this fold
    enc_path = f"{MODEL_DIR}/encoder_{fold_name}.weights.h5"
    dec_path = f"{MODEL_DIR}/decoder_{fold_name}.weights.h5"
    
    print(f"Loading encoder: {enc_path}")
    print(f"Loading decoder: {dec_path}")
    cvae.encoder.load_weights(enc_path)
    cvae.decoder.load_weights(dec_path)
    
    encoder = cvae.encoder
    decoder = cvae.decoder
    
    # --- PHASE 1: ENCODE ---
    print("  Extracting latent representations (z)...")
    z_mean, _, _, _ = encoder.predict(X_val_exp, batch_size=256, verbose=0)
    
    # --- PHASE 2: DECODE (6 Times) ---
    print("  Computing 6-class reconstruction MSE...")
    mse_scores = np.zeros((n_samples, 6), dtype=np.float32)
    
    for class_idx in range(6):
        requested_classes = tf.convert_to_tensor(
            tf.keras.utils.to_categorical([class_idx] * n_samples, num_classes=6),
            dtype=tf.float32
        )
        
        # "If this spectrum was Class X, what should it look like?"
        reconstructed_spectra = decoder.predict([z_mean, requested_classes], batch_size=256, verbose=0)
        reconstructed_spectra = np.squeeze(reconstructed_spectra, axis=-1)
        
        # Calculate Mean Squared Error across all 1738 bands
        mse = np.mean((X_val - reconstructed_spectra)**2, axis=1)
        mse_scores[:, class_idx] = mse
        
    # --- PHASE 3: CLASSIFY ---
    # The winner is the class with the lowest MSE (best reconstruction)
    y_pred_cbr = np.argmin(mse_scores, axis=1)
    
    acc = accuracy_score(y_val, y_pred_cbr)
    cbr_results['Classification_by_Recon'].append(acc)
    print(f" {fold_name} Classification by Reconstruction Accuracy: {acc:.4f}")
    
    # Cleanup for next fold
    del cvae, encoder, decoder, X_val, y_val, X_val_exp, z_mean, mse_scores
    tf.keras.backend.clear_session()
    gc.collect()

# --- Print Final Summary ---
print("\n" + "="*60)
print(" === CLASSIFICATION BY RECONSTRUCTION COMPLETE ===")
print("="*60)
mean_acc = np.mean(cbr_results['Classification_by_Recon'])
std_acc = np.std(cbr_results['Classification_by_Recon'])
print(f"Classification by Reconstruction: {mean_acc:.4f} ± {std_acc:.4f}")
print("="*60)


EXPERIMENT SETUP

Working directory: c:\Users\Andrei\incdsb\New folder\glioma-synthetic-data

 Loaded configuration: exp_104
   Description: TSAR Regimen: Fingerprint crop (700-1800), patient cap (500), and dynamic Raman augmentation

 Output directory: C:\Users\Andrei\incdsb\New folder\glioma-synthetic-data\results\results
  Experiment already registered (re-running)

--------------------------------------------------------------------------------
Configuration Summary:
--------------------------------------------------------------------------------
  Preprocessing:
    - AirPLS lambda: 100
    - Signal length: 1738
    - Crop region: [700, 1800]

  CV Strategy:
    - Split source: data/splits/strict_splits.json
    - Use group-aware CV: True
    - Hold out test set: True

  Models:
    - RF n_estimators: 100
    - SVM C: 1.0
    - CVAE epochs: 70


✓ Config loaded: exp_104
Data path: results/exp_102/data/glioma_dataset.h5
Model weights dir: results/exp_104/models/cvae
CVAE Config lo

In [2]:
%reset -f